# Введение в MapReduce модель на Python


In [63]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [64]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [65]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [66]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [67]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [68]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [69]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [70]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [71]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [72]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [73]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [74]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных. 

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [75]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*
 
mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL 

In [76]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str
    
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)
 
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication 

In [77]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])
 
def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])
      
output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.2964828267284383)),
 (1, np.float64(2.2964828267284383)),
 (2, np.float64(2.2964828267284383)),
 (3, np.float64(2.2964828267284383)),
 (4, np.float64(2.2964828267284383))]

## Inverted index 

In [78]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)
      
def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)
 
def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [79]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [80]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()
      
def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]
 
def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers
  
def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)
  
  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*
 
flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount 

In [81]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)
      
  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)
  
# try to set COMBINER=REDUCER and look at the number of values sent over the network 
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None) 
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('it', 18)]),
 (1, [('is', 18), ('what', 10)])]

## TeraSort

In [82]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for value in split:
        yield (value, None)
      
  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])
    
def MAP(value:int, _):
  yield (value, None)
  
def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)
  
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.028057841189348398)),
   (None, np.float64(0.08278723327292326)),
   (None, np.float64(0.09510445437241688)),
   (None, np.float64(0.10845305199137611)),
   (None, np.float64(0.11279975277955034)),
   (None, np.float64(0.14647603118223118)),
   (None, np.float64(0.1742422358822162)),
   (None, np.float64(0.1767499693482386)),
   (None, np.float64(0.225491548063539)),
   (None, np.float64(0.28441307372654123)),
   (None, np.float64(0.2921268589965015)),
   (None, np.float64(0.3239506685033806)),
   (None, np.float64(0.3409599456175193)),
   (None, np.float64(0.37282073005607963)),
   (None, np.float64(0.40307818409037866)),
   (None, np.float64(0.48953416859493215)),
   (None, np.float64(0.4943622260307222))]),
 (1,
  [(None, np.float64(0.617514169053263)),
   (None, np.float64(0.6214188546445593)),
   (None, np.float64(0.6436509701615025)),
   (None, np.float64(0.6668740149448226)),
   (None, np.float64(0.7149558441607348)),
   (None, np.float64(0.7385973950

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [83]:
# groupByKey via sorting
# Input: (key, value). Output: (key, [values]).

def groupbykey_sorted(iterable):
  sorted_items = sorted(iterable, key=lambda kv: kv[0])
  current_key = None
  current_values = []
  has_key = False

  for key, value in sorted_items:
    if (not has_key) or key != current_key:
      if has_key:
        yield (current_key, current_values)
      current_key = key
      current_values = [value]
      has_key = True
    else:
      current_values.append(value)

  if has_key:
    yield (current_key, current_values)

pairs = [('b', 1), ('a', 10), ('b', 2), ('c', 7), ('a', 11)]
result = list(groupbykey_sorted(pairs))
expected = [('a', [10, 11]), ('b', [1, 2]), ('c', [7])]
assert result == expected
result


[('a', [10, 11]), ('b', [1, 2]), ('c', [7])]

### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [84]:
# Distributed distinct.
# MAP uses the value as a key. REDUCE emits one value per group.

values = ['cat', 'dog', 'cat', 'bird', 'dog', 'ant', 'bird', 'cat']
maps = 3
reducers = 2

def INPUTFORMAT():
  split_size = int(np.ceil(len(values) / maps))

  def RECORDREADER(split, offset):
    for local_id, value in enumerate(split):
      yield (offset + local_id, value)

  for start in range(0, len(values), split_size):
    yield RECORDREADER(values[start:start + split_size], start)

def MAP(record_id, value):
  yield (value, None)

def COMBINER_DISTINCT(value, markers):
  yield (value, None)

def REDUCE(value, markers):
  yield (None, value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=COMBINER_DISTINCT)
distinct_values = sorted(value for _, partition in partitioned_output for _, value in partition)

assert distinct_values == sorted(set(values))
distinct_values


7 key-value pairs were sent over a network.


['ant', 'bird', 'cat', 'dog']

#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [85]:
# Grouping and aggregation: key = a, value = b.

rows = [
  ('x', 10, 'p'),
  ('y', 3, 'q'),
  ('x', 7, 'r'),
  ('y', 5, 's'),
  ('z', 100, 't'),
]

def RECORDREADER():
  for row_id, row in enumerate(rows):
    yield (row_id, row)

def MAP(row_id, row):
  a, b, c = row
  yield (a, b)

def REDUCE(a, values):
  yield (a, sum(values))

result = dict(MapReduce(RECORDREADER, MAP, REDUCE))
expected = {}
for a, b, c in rows:
  expected[a] = expected.get(a, 0) + b

assert result == expected
result


{'x': 17, 'y': 8, 'z': 100}

# 

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [86]:
# Matrix-vector when the vector is read as data.
# Job 1 joins by j. Job 2 sums partial products by i.

import numpy as np

mat = np.random.rand(5, 4)
vec = np.random.rand(4)

def RECORDREADER_JOIN():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield (('M', i, j), mat[i, j])
  for j in range(vec.shape[0]):
    yield (('x', j), vec[j])

def MAP_JOIN(k1, v1):
  if k1[0] == 'M':
    _, i, j = k1
    yield (j, ('M', i, v1))
  else:
    _, j = k1
    yield (j, ('x', v1))

def REDUCE_JOIN(j, values):
  matrix_items = []
  vector_value = None

  for value in values:
    if value[0] == 'M':
      _, i, m_ij = value
      matrix_items.append((i, m_ij))
    else:
      _, vector_value = value

  if vector_value is not None:
    for i, m_ij in matrix_items:
      yield (i, m_ij * vector_value)

partial_products = list(MapReduce(RECORDREADER_JOIN, MAP_JOIN, REDUCE_JOIN))

def RECORDREADER_SUM():
  return partial_products

def MAP_SUM(i, product):
  yield (i, product)

def REDUCE_SUM(i, products):
  yield (i, sum(products))

result = dict(MapReduce(RECORDREADER_SUM, MAP_SUM, REDUCE_SUM))
solution = np.array([result[i] for i in range(mat.shape[0])])
reference_solution = mat.dot(vec)

assert np.allclose(solution, reference_solution)
solution


array([1.20325564, 1.16603622, 1.11585085, 0.27851774, 1.3730288 ])

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$. 





In [87]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [88]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

# One matrix is available in mapper memory.
# MAP: ((j, k), n_jk) -> ((i, k), m_ij * n_jk)
# REDUCE sums by (i, k).

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])
      
def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):
    yield ((i, k), small_mat[i, j] * w)
  
def REDUCE(key, values):
  (i, k) = key
  yield ((i, k), sum(values))


Проверьте своё решение

In [89]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat) 
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [90]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [91]:
# Both matrices are generated by RECORDREADER.
# First join by j, then sum partial products by (i, k).

I = 2
J = 3
K = 5
left_mat = np.random.rand(I, J)
right_mat = np.random.rand(J, K)

def RECORDREADER_JOIN():
  for i in range(left_mat.shape[0]):
    for j in range(left_mat.shape[1]):
      yield (('M', i, j), left_mat[i, j])
  for j in range(right_mat.shape[0]):
    for k in range(right_mat.shape[1]):
      yield (('N', j, k), right_mat[j, k])

def MAP_JOIN(k1, v1):
  tag = k1[0]
  if tag == 'M':
    _, i, j = k1
    yield (j, ('M', i, v1))
  else:
    _, j, k = k1
    yield (j, ('N', k, v1))

def REDUCE_JOIN(j, values):
  left_items = []
  right_items = []

  for value in values:
    if value[0] == 'M':
      _, i, m_ij = value
      left_items.append((i, m_ij))
    else:
      _, k, n_jk = value
      right_items.append((k, n_jk))

  for i, m_ij in left_items:
    for k, n_jk in right_items:
      yield ((i, k), m_ij * n_jk)

partial_products = list(MapReduce(RECORDREADER_JOIN, MAP_JOIN, REDUCE_JOIN))

def RECORDREADER_SUM():
  return partial_products

def MAP_SUM(key, product):
  yield (key, product)

def REDUCE_SUM(key, products):
  yield (key, sum(products))

solution_items = list(MapReduce(RECORDREADER_SUM, MAP_SUM, REDUCE_SUM))
solution_mat = asmatrix(solution_items)
reference_solution = np.matmul(left_mat, right_mat)

assert np.allclose(solution_mat, reference_solution)
solution_mat


array([[0.25382665, 0.68446549, 0.76308382, 0.22982382, 0.77898582],
       [0.86896981, 0.57661211, 1.21608911, 0.56084433, 0.64648936]])

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER. 

In [92]:
# Distributed matrix multiplication.
# Readers are separate, but the logic is the same: join by j, then sum by (i, k).

I = 2
J = 3
K = 5
left_mat = np.random.rand(I, J)
right_mat = np.random.rand(J, K)
maps = 2
reducers = 2

def INPUTFORMAT_JOIN():
  def RECORDREADER_LEFT():
    for i in range(left_mat.shape[0]):
      for j in range(left_mat.shape[1]):
        yield (('M', i, j), left_mat[i, j])

  def RECORDREADER_RIGHT():
    for j in range(right_mat.shape[0]):
      for k in range(right_mat.shape[1]):
        yield (('N', j, k), right_mat[j, k])

  yield RECORDREADER_LEFT()
  yield RECORDREADER_RIGHT()

def MAP_JOIN(k1, v1):
  if k1[0] == 'M':
    _, i, j = k1
    yield (j, ('M', i, v1))
  else:
    _, j, k = k1
    yield (j, ('N', k, v1))

def REDUCE_JOIN(j, values):
  left_items = []
  right_items = []

  for value in values:
    if value[0] == 'M':
      _, i, m_ij = value
      left_items.append((i, m_ij))
    else:
      _, k, n_jk = value
      right_items.append((k, n_jk))

  for i, m_ij in left_items:
    for k, n_jk in right_items:
      yield ((i, k), m_ij * n_jk)

join_output = MapReduceDistributed(INPUTFORMAT_JOIN, MAP_JOIN, REDUCE_JOIN)
partial_products = [(key, product) for _, partition in join_output for key, product in partition]

def INPUTFORMAT_SUM():
  def RECORDREADER_PRODUCTS():
    for item in partial_products:
      yield item

  yield RECORDREADER_PRODUCTS()

def MAP_SUM(key, product):
  yield (key, product)

def REDUCE_SUM(key, products):
  yield (key, sum(products))

sum_output = MapReduceDistributed(INPUTFORMAT_SUM, MAP_SUM, REDUCE_SUM)
solution_items = [(key, value) for _, partition in sum_output for key, value in partition]
solution_mat = asmatrix(solution_items)
reference_solution = np.matmul(left_mat, right_mat)

assert np.allclose(solution_mat, reference_solution)
solution_mat


21 key-value pairs were sent over a network.
30 key-value pairs were sent over a network.


array([[1.16023119, 1.09444961, 1.34980561, 0.6704605 , 1.32802184],
       [0.91978657, 0.9432511 , 1.15349286, 0.44382254, 1.11520621]])

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [93]:
# Several readers per matrix.
# Each element must appear in exactly one reader.
# Random subsets are unsafe unless they guarantee full coverage without duplicates.

I = 4
J = 3
K = 5
left_mat = np.random.rand(I, J)
right_mat = np.random.rand(J, K)
maps_per_matrix = 2
reducers = 3

def chunks(items, chunk_count):
  chunk_size = int(np.ceil(len(items) / chunk_count))
  for start in range(0, len(items), chunk_size):
    yield items[start:start + chunk_size]

left_items = [(('M', i, j), left_mat[i, j]) for i in range(I) for j in range(J)]
right_items = [(('N', j, k), right_mat[j, k]) for j in range(J) for k in range(K)]

def INPUTFORMAT_JOIN():
  def make_reader(split):
    def RECORDREADER():
      for item in split:
        yield item
    return RECORDREADER()

  for split in chunks(left_items, maps_per_matrix):
    yield make_reader(split)
  for split in chunks(right_items, maps_per_matrix):
    yield make_reader(split)

join_output = MapReduceDistributed(INPUTFORMAT_JOIN, MAP_JOIN, REDUCE_JOIN)
partial_products = [(key, product) for _, partition in join_output for key, product in partition]

def INPUTFORMAT_SUM():
  split_size = int(np.ceil(len(partial_products) / reducers))

  def make_reader(split):
    def RECORDREADER():
      for item in split:
        yield item
    return RECORDREADER()

  for start in range(0, len(partial_products), split_size):
    yield make_reader(partial_products[start:start + split_size])

sum_output = MapReduceDistributed(INPUTFORMAT_SUM, MAP_SUM, REDUCE_SUM)
solution_items = [(key, value) for _, partition in sum_output for key, value in partition]
solution_mat = asmatrix(solution_items)
reference_solution = np.matmul(left_mat, right_mat)

assert np.allclose(solution_mat, reference_solution)
solution_mat


27 key-value pairs were sent over a network.
60 key-value pairs were sent over a network.


array([[1.02944309, 0.91992026, 1.14421641, 1.25421925, 1.30199016],
       [1.12461624, 0.89175346, 1.0333471 , 1.26135505, 1.40480578],
       [0.94983173, 0.97475658, 0.8345571 , 1.01871557, 1.07846482],
       [0.81792342, 0.73250443, 0.70098952, 0.87945227, 0.9696739 ]])